# 🏎️ Fabric Racing Game v3

Race through 10 Fabric-themed tracks! Collect ⭐ stars, avoid 🐛 bugs.

## How to Play (3 Easy Steps)
1. **Run Cell 1** → Configure connection string  
2. **Run Cell 2** → Play the game! At end, click "Download Events"
3. **Run Cell 3** → Upload the downloaded file → Telemetry sent!

**No copy/paste needed!** Just download a file and upload it.

In [ ]:
# ⚙️ CELL 1: CONFIGURATION
# Get Connection String from: Eventstream → Custom Endpoint → Keys

CONNECTION_STRING = "Endpoint=sb://YOUR_NAMESPACE.servicebus.windows.net/;SharedAccessKeyName=YOUR_KEY_NAME;SharedAccessKey=YOUR_KEY;EntityPath=YOUR_EVENTHUB"
PLAYER_NAME = "Player1"

# Install SDK
%pip install azure-eventhub ipywidgets -q

print(f"✅ Ready! Player: {PLAYER_NAME}")

In [ ]:
# 🎮 CELL 2: PLAY THE GAME!
from IPython.display import display, HTML
import uuid

session_id = str(uuid.uuid4())

game_html = f'''
<!DOCTYPE html>
<html>
<head>
<style>
* {{ margin: 0; padding: 0; box-sizing: border-box; }}
#game-wrapper {{ 
    width: 100%; display: flex; flex-direction: column; align-items: center;
    font-family: 'Segoe UI', Arial, sans-serif;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    padding: 20px; border-radius: 15px;
}}
#game-container {{ 
    position: relative; width: 600px; height: 500px;
    background: #2d3436; border-radius: 10px; overflow: hidden;
    box-shadow: 0 10px 30px rgba(0,0,0,0.5);
}}
canvas {{ display: block; }}
#hud {{
    position: absolute; top: 10px; left: 10px; right: 50px;
    display: flex; justify-content: space-between;
    color: white; font-size: 14px; text-shadow: 2px 2px 4px rgba(0,0,0,0.8); z-index: 10;
}}
#hud-left, #hud-right {{ background: rgba(0,0,0,0.7); padding: 10px 14px; border-radius: 8px; }}
#lives {{ color: #ff6b6b; font-size: 18px; letter-spacing: 3px; }}
#progress-bar {{
    position: absolute; right: 15px; top: 80px; bottom: 80px; width: 25px;
    background: rgba(0,0,0,0.7); border-radius: 12px; z-index: 10;
}}
#progress-fill {{ position: absolute; bottom: 0; width: 100%; background: linear-gradient(to top, #00b894, #55efc4); border-radius: 10px; transition: height 0.15s; }}
#progress-car {{ position: absolute; left: -8px; width: 40px; text-align: center; font-size: 18px; transition: bottom 0.15s; }}
#start-btn {{
    position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%);
    padding: 25px 60px; font-size: 32px;
    background: linear-gradient(135deg, #00b894, #00a085);
    color: white; border: none; border-radius: 20px; cursor: pointer; z-index: 100;
}}
#level-info, #game-over {{
    position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%);
    background: rgba(0,0,0,0.95); padding: 40px; border-radius: 15px;
    color: white; text-align: center; z-index: 100; display: none;
}}
#event-count {{ position: absolute; bottom: 5px; left: 5px; font-size: 11px; color: #888; }}
.multiplier {{ color: #ffd700; }}
#download-btn {{
    margin-top: 15px; padding: 12px 30px;
    background: linear-gradient(135deg, #3498db, #2980b9);
    color: white; border: none; border-radius: 10px; cursor: pointer;
    font-size: 16px;
}}
#download-btn:hover {{ background: linear-gradient(135deg, #2980b9, #1a5276); }}
</style>
</head>
<body>
<div id="game-wrapper" tabindex="0">
    <div id="game-container">
        <canvas id="gameCanvas" width="600" height="500"></canvas>
        <div id="hud">
            <div id="hud-left">
                <div style="font-size:16px; color:#ffd700;">🏎️ <span id="level-name">Lakehouse Lane</span></div>
                <div>Level: <span id="level">1</span>/10</div>
                <div id="lives">❤️❤️❤️</div>
            </div>
            <div id="hud-right">
                <div style="font-size:18px;">Score: <span id="score">0</span></div>
                <div>Multiplier: <span id="multiplier" class="multiplier">x1</span></div>
                <div>Target: <span id="target" style="color:#ffd700;">500</span></div>
            </div>
        </div>
        <div id="progress-bar">
            <div id="progress-fill" style="height: 0%"></div>
            <div id="progress-car">🏎️</div>
        </div>
        <button id="start-btn" onclick="startGame()">▶ START</button>
        <div id="level-info">
            <h2 id="level-title">Level Complete!</h2>
            <p id="level-message"></p>
            <button id="continue-btn" onclick="continueGame()" style="margin-top:15px; padding:12px 30px; background:#00b894; color:white; border:none; border-radius:10px; cursor:pointer;">Continue</button>
        </div>
        <div id="game-over">
            <h2 style="color:#e74c3c;">💀 GAME OVER 💀</h2>
            <p>Final Score: <b id="final-score">0</b></p>
            <p>Reached Level: <span id="final-level">1</span></p>
            <button id="download-btn" onclick="downloadEvents()">📥 Download Events</button>
            <p style="margin-top:10px; font-size:12px; color:#888;">Then run Cell 3 to send telemetry</p>
            <button onclick="restartGame()" style="margin-top:10px; padding:10px 25px; background:#e74c3c; color:white; border:none; border-radius:10px; cursor:pointer;">🔄 Play Again</button>
        </div>
        <div id="event-count">Events: 0</div>
    </div>
</div>

<script>
window.gameEvents = [];

const LEVELS = [
    {{ name: "Lakehouse Lane", target: 500, stars: 12, bugs: 5, length: 1500, speed: 4 }},
    {{ name: "Pipeline Pass", target: 800, stars: 14, bugs: 7, length: 1800, speed: 4.5 }},
    {{ name: "Warehouse Way", target: 1200, stars: 16, bugs: 9, length: 2000, speed: 5 }},
    {{ name: "Dataflow Drive", target: 1600, stars: 18, bugs: 11, length: 2200, speed: 5.5 }},
    {{ name: "Notebook Narrows", target: 2000, stars: 20, bugs: 14, length: 2500, speed: 6 }},
    {{ name: "Eventhouse Express", target: 2500, stars: 22, bugs: 17, length: 2800, speed: 6.5 }},
    {{ name: "Shortcut Sprint", target: 3000, stars: 24, bugs: 20, length: 3000, speed: 7 }},
    {{ name: "Capacity Canyon", target: 3500, stars: 26, bugs: 24, length: 3300, speed: 7.5 }},
    {{ name: "OneLake Overdrive", target: 4000, stars: 28, bugs: 28, length: 3600, speed: 8 }},
    {{ name: "Spark Summit", target: 5000, stars: 32, bugs: 32, length: 4000, speed: 9 }}
];

const canvas = document.getElementById("gameCanvas");
const ctx = canvas.getContext("2d");
const W = 600, H = 500;

let gameRunning = false, currentLevel = 0, levelScore = 0, totalScore = 0, multiplier = 1;
let consecutiveStars = 0, distance = 0, raceFinished = false, lives = 3;
let car = {{ x: W/2, y: H - 80, width: 40, height: 60, speed: 0 }};
let stars = [], bugs = [], roadOffset = 0, keys = {{}};

const sessionId = "{session_id}";
const playerName = "{PLAYER_NAME}";

function downloadEvents() {{
    const json = JSON.stringify(window.gameEvents, null, 2);
    const blob = new Blob([json], {{ type: 'application/json' }});
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    a.download = 'racing_events.json';
    a.click();
    URL.revokeObjectURL(url);
    document.getElementById('download-btn').textContent = '✅ Downloaded!';
    document.getElementById('download-btn').style.background = '#27ae60';
}}

function recordEvent(eventType, extraData = {{}}) {{
    const event = {{
        EventId: crypto.randomUUID(),
        Timestamp: new Date().toISOString(),
        GameSessionId: sessionId,
        PlayerId: playerName,
        EventType: eventType,
        Level: currentLevel + 1,
        LevelName: LEVELS[currentLevel]?.name || "Unknown",
        Score: levelScore,
        TotalScore: totalScore,
        Lives: lives,
        Multiplier: multiplier,
        ...extraData
    }};
    window.gameEvents.push(event);
    document.getElementById("event-count").textContent = "Events: " + window.gameEvents.length;
}}

function updateLivesDisplay() {{ document.getElementById("lives").textContent = "❤️".repeat(lives) + "🖤".repeat(3-lives); }}

function initLevel() {{
    const level = LEVELS[currentLevel];
    distance = 0; levelScore = 0; multiplier = 1; consecutiveStars = 0; raceFinished = false;
    car.x = W/2; car.speed = level.speed;
    stars = []; bugs = [];
    
    for (let i = 0; i < level.stars; i++) {{
        stars.push({{ x: 120 + Math.random() * 360, y: -(200 + i * (level.length/level.stars)), collected: false }});
    }}
    for (let i = 0; i < level.bugs; i++) {{
        bugs.push({{ x: 120 + Math.random() * 360, y: -(250 + i * (level.length/level.bugs)), hit: false }});
    }}
    
    document.getElementById("level-info").style.display = "none";
    updateLivesDisplay();
    updateHUD();
    recordEvent("LevelStart", {{ TargetScore: level.target }});
}}

function updateHUD() {{
    const level = LEVELS[currentLevel];
    document.getElementById("level").textContent = currentLevel + 1;
    document.getElementById("level-name").textContent = level.name;
    document.getElementById("score").textContent = levelScore;
    document.getElementById("multiplier").textContent = "x" + multiplier;
    document.getElementById("target").textContent = level.target;
    const progress = Math.min(100, (distance / level.length) * 100);
    document.getElementById("progress-fill").style.height = progress + "%";
    document.getElementById("progress-car").style.bottom = "calc(" + progress + "% - 10px)";
}}

function drawRoad() {{
    ctx.fillStyle = "#1a1a3e"; ctx.fillRect(0, 0, W, H);
    ctx.fillStyle = "#2d3436"; ctx.fillRect(100, 0, W-200, H);
    ctx.fillStyle = "#3498db"; ctx.fillRect(95, 0, 8, H); ctx.fillRect(W-103, 0, 8, H);
    ctx.strokeStyle = "rgba(255,255,255,0.4)"; ctx.setLineDash([40, 25]); ctx.lineWidth = 4;
    ctx.beginPath();
    for (let y = (roadOffset % 65) - 65; y < H + 65; y += 65) {{ ctx.moveTo(W/2, y); ctx.lineTo(W/2, y + 40); }}
    ctx.stroke(); ctx.setLineDash([]);
    
    const finishY = H - (distance - (LEVELS[currentLevel].length - 100));
    if (finishY > -50 && finishY < H + 50) {{
        for (let x = 100; x < W - 100; x += 20) {{
            for (let row = 0; row < 2; row++) {{
                ctx.fillStyle = ((x/20) + row) % 2 === 0 ? "#fff" : "#000";
                ctx.fillRect(x, finishY + row*20, 20, 20);
            }}
        }}
        ctx.fillStyle = "#ffd700"; ctx.font = "bold 22px Arial"; ctx.textAlign = "center";
        ctx.fillText("🏁 FINISH 🏁", W/2, finishY - 15); ctx.textAlign = "left";
    }}
}}

function drawCar() {{
    ctx.save(); ctx.translate(car.x, car.y);
    ctx.fillStyle = "#e74c3c";
    ctx.beginPath(); ctx.roundRect(-car.width/2, -car.height/2, car.width, car.height, 8); ctx.fill();
    ctx.fillStyle = "#74b9ff"; ctx.fillRect(-14, -car.height/2 + 8, 28, 16);
    ctx.fillStyle = "#fff"; ctx.fillRect(-3, -car.height/2, 6, car.height);
    ctx.fillStyle = "#222";
    ctx.fillRect(-car.width/2 - 4, -car.height/2 + 8, 6, 16);
    ctx.fillRect(car.width/2 - 2, -car.height/2 + 8, 6, 16);
    ctx.fillRect(-car.width/2 - 4, car.height/2 - 24, 6, 16);
    ctx.fillRect(car.width/2 - 2, car.height/2 - 24, 6, 16);
    ctx.restore();
}}

function drawStars() {{
    stars.forEach(star => {{
        if (!star.collected) {{
            const screenY = star.y + distance;
            if (screenY > -40 && screenY < H + 40) {{ ctx.font = "32px Arial"; ctx.fillText("⭐", star.x - 16, screenY + 12); }}
        }}
    }});
}}

function drawBugs() {{
    bugs.forEach(bug => {{
        if (!bug.hit) {{
            const screenY = bug.y + distance;
            if (screenY > -40 && screenY < H + 40) {{ ctx.font = "30px Arial"; ctx.fillText("🐛", bug.x - 15, screenY + 10); }}
        }}
    }});
}}

function checkCollisions() {{
    if (raceFinished) return;
    stars.forEach(star => {{
        if (!star.collected) {{
            const screenY = star.y + distance;
            if (Math.abs(car.x - star.x) < 40 && Math.abs(car.y - screenY) < 45) {{
                star.collected = true; consecutiveStars++;
                multiplier = Math.min(10, Math.floor(consecutiveStars / 3) + 1);
                const points = 50 * multiplier; levelScore += points;
                recordEvent("StarCollected", {{ Points: points }});
            }}
        }}
    }});
    bugs.forEach(bug => {{
        if (!bug.hit) {{
            const screenY = bug.y + distance;
            if (Math.abs(car.x - bug.x) < 35 && Math.abs(car.y - screenY) < 40) {{
                bug.hit = true; levelScore = Math.max(0, levelScore - 30);
                consecutiveStars = 0; multiplier = 1;
                recordEvent("BugHit", {{ PenaltyPoints: 30 }});
            }}
        }}
    }});
    updateHUD();
}}

function checkFinish() {{
    const level = LEVELS[currentLevel];
    if (!raceFinished && distance >= level.length) {{
        raceFinished = true; gameRunning = false;
        const success = levelScore >= level.target;
        const levelInfo = document.getElementById("level-info");
        
        if (success) {{
            totalScore += levelScore;
            recordEvent("LevelComplete", {{ Success: true }});
            if (currentLevel >= LEVELS.length - 1) {{
                document.getElementById("level-title").textContent = "🏆 CHAMPION! 🏆";
                document.getElementById("level-message").innerHTML = "All tracks done!<br>Score: " + totalScore + "<br><br>";
                document.getElementById("level-message").innerHTML += '<button onclick="downloadEvents()" style="padding:12px 30px; background:#3498db; color:white; border:none; border-radius:10px; cursor:pointer;">📥 Download Events</button>';
                recordEvent("GameComplete", {{ FinalScore: totalScore }});
                document.getElementById("continue-btn").style.display = "none";
            }} else {{
                document.getElementById("level-title").textContent = "✅ Level Complete!";
                document.getElementById("level-message").innerHTML = "Score: " + levelScore + " / " + level.target;
            }}
            levelInfo.style.display = "block";
        }} else {{
            lives--; updateLivesDisplay();
            recordEvent("LevelFailed", {{ Success: false }});
            if (lives <= 0) {{
                document.getElementById("final-score").textContent = totalScore;
                document.getElementById("final-level").textContent = currentLevel + 1;
                document.getElementById("game-over").style.display = "block";
                recordEvent("GameOver", {{ FinalScore: totalScore }});
            }} else {{
                document.getElementById("level-title").textContent = "❌ Try Again!";
                document.getElementById("level-message").innerHTML = "Lives: " + lives;
                levelInfo.style.display = "block";
            }}
        }}
    }}
}}

function gameLoop() {{
    if (!gameRunning) return;
    if (keys.ArrowLeft || keys.KeyA) car.x = Math.max(130, car.x - 7);
    if (keys.ArrowRight || keys.KeyD) car.x = Math.min(W - 130, car.x + 7);
    distance += car.speed;
    roadOffset += car.speed;
    drawRoad(); drawStars(); drawBugs(); drawCar();
    checkCollisions(); checkFinish();
    requestAnimationFrame(gameLoop);
}}

function startGame() {{
    document.getElementById("start-btn").style.display = "none";
    window.gameEvents = [];
    currentLevel = 0; totalScore = 0; lives = 3;
    recordEvent("GameStart");
    initLevel();
    gameRunning = true;
    gameLoop();
}}

function continueGame() {{
    const level = LEVELS[currentLevel];
    if (levelScore >= level.target) currentLevel++;
    initLevel();
    gameRunning = true;
    gameLoop();
}}

function restartGame() {{
    document.getElementById("game-over").style.display = "none";
    document.getElementById("download-btn").textContent = '📥 Download Events';
    document.getElementById("download-btn").style.background = 'linear-gradient(135deg, #3498db, #2980b9)';
    startGame();
}}

document.addEventListener("keydown", e => {{ keys[e.code] = true; if (["ArrowLeft","ArrowRight","ArrowUp","ArrowDown"].includes(e.key)) e.preventDefault(); }});
document.addEventListener("keyup", e => {{ keys[e.code] = false; }});
</script>
</body>
</html>
'''

display(HTML(game_html))
print("🎮 Use ← → arrow keys to steer. Collect ⭐, avoid 🐛!")

In [ ]:
# 📤 CELL 3: UPLOAD & SEND TELEMETRY
# Upload the racing_events.json file you downloaded from the game

import ipywidgets as widgets
from IPython.display import display, HTML
import json
from azure.eventhub import EventHubProducerClient, EventData

# Create file upload widget
upload = widgets.FileUpload(accept='.json', multiple=False, description='📁 Upload JSON')
output = widgets.Output()
send_btn = widgets.Button(description='📡 Send Telemetry', button_style='success', disabled=True)
status = widgets.HTML('<p style="color:#888;">⬆️ Upload racing_events.json from your Downloads folder</p>')

def on_upload(change):
    if upload.value:
        file_info = list(upload.value.values())[0] if isinstance(upload.value, dict) else upload.value[0]
        content = file_info['content']
        events = json.loads(content.tobytes().decode('utf-8'))
        upload.events_data = events
        status.value = f'<p style="color:#2ecc71;">✅ Loaded {len(events)} events. Click Send!</p>'
        send_btn.disabled = False

def on_send(b):
    if hasattr(upload, 'events_data') and upload.events_data:
        events = upload.events_data
        with output:
            output.clear_output()
            try:
                producer = EventHubProducerClient.from_connection_string(CONNECTION_STRING)
                sent = 0
                with producer:
                    for i in range(0, len(events), 100):
                        batch = producer.create_batch()
                        for event in events[i:i+100]:
                            try:
                                batch.add(EventData(json.dumps(event)))
                                sent += 1
                            except ValueError:
                                producer.send_batch(batch)
                                batch = producer.create_batch()
                                batch.add(EventData(json.dumps(event)))
                                sent += 1
                        producer.send_batch(batch)
                print(f"🎉 SUCCESS! Sent {sent} events to Eventhouse!")
                status.value = f'<p style="color:#27ae60; font-size:18px;">🎉 Sent {sent} events!</p>'
            except Exception as e:
                print(f"❌ Error: {e}")
                status.value = f'<p style="color:#e74c3c;">❌ Error: {e}</p>'

upload.observe(on_upload, names='value')
send_btn.on_click(on_send)

display(widgets.VBox([
    widgets.HTML('<h3>📡 Send Game Telemetry</h3>'),
    upload,
    status,
    send_btn,
    output
]))